In [5]:
import matplotlib.pyplot as plt
import pandas as pd
%run functions_dbs.py

fs = 10
%matplotlib

Using matplotlib backend: MacOSX


In [20]:
file = 'Complete_T1.xlsx'
ddata = pd.read_excel(file, sheet_name=None)

# get the EP data (ideally all profiles)
ls_ep = list()
[ls_ep.append(i) for i in ddata.keys() if 'EP' in i or 'Ep' in i or 'ep' in i]
key = None
for x in ls_ep:
    if 'all' in x:
        key = x
if key:
    pass
else:
    key = ls_ep[0]
df_ep = ddata[key]

# get core - sample order
order = df_ep[['Nr','Core']].drop_duplicates().values

# split into cores
gb = df_ep.groupby(by='Core')
dic_ep = dict(map(lambda x: (x, gb.get_group(x)), gb.groups))

# split one core into sample-Nr
for c in dic_ep.keys():
    gp = dic_ep[c].groupby(by='Nr')
    dic_epS = dict(map(lambda x: (x, gp.get_group(x)), gp.groups))
    dic_ep[c] = dic_epS

for c in dic_ep.keys():
    for s in dic_ep[c].keys():
        d = dic_ep[c][s][['Depth', 'EP_mV']].set_index('Depth')
        dic_ep[c][s] = d

df_ep = pd.concat(dict(map(lambda c: (c, pd.concat(dic_ep[c], axis=1)), dic_ep.keys())), axis=1)

# bring to correct order
ls = list()
[ls.append(o[1]) for o in order if o[1] not in ls]

new_cols = df_ep.columns.reindex(ls, level=0)
df_ep = df_ep.reindex(columns=new_cols[0])
df_ep

35                                          26             \
                 13         14         15         16         17         18   
              EP_mV      EP_mV      EP_mV      EP_mV      EP_mV      EP_mV   
Depth                                                                        
-2000.0         NaN        NaN        NaN        NaN        NaN        NaN   
-1500.0         NaN        NaN        NaN        NaN        NaN        NaN   
-1000.0   39.589539  34.382629  34.489746  33.480072  33.787689  33.512726   
-900.0          NaN        NaN        NaN        NaN        NaN        NaN   
-700.0          NaN        NaN        NaN        NaN        NaN        NaN   
...             ...        ...        ...        ...        ...        ...   
 29400.0        NaN        NaN        NaN        NaN        NaN        NaN   
 29500.0  36.677856  35.592346  35.184021  33.962708  29.951019  50.415802   
 29700.0        NaN        NaN        NaN        NaN        NaN        NaN   
 30000.0  36.549988  35.628967  35.198669  33.966980  30.097809  53.135986   
 NaN            NaN        NaN        NaN        NaN        NaN        NaN   

                                       16             ...         15  \
                 19         20         27         28  ...         39   
              EP_mV      EP_mV      EP_mV      EP_mV  ...      EP_mV   
Depth                                                 ...              
-2000.0         NaN        NaN        NaN        NaN  ...  32.801514   
-1500.0         NaN        NaN        NaN        NaN  ...  32.810059   
-1000.0   33.409424  33.330536  34.272003  33.107910  ...  32.828369   
-900.0          NaN        NaN        NaN        NaN  ...        NaN   
-700.0          NaN        NaN        NaN  33.102722  ...        NaN   
...             ...        ...        ...        ...  ...        ...   
 29400.0        NaN        NaN        NaN        NaN  ...        NaN   
 29500.0  28.848114  32.600098  31.597290  30.440674  ...  33.178406   
 29700.0        NaN        NaN        NaN        NaN  ...        NaN   
 30000.0  28.378143  32.553558  31.301270  30.227356  ...  33.155518   
 NaN            NaN        NaN        NaN        NaN  ...        NaN   

                                                  23                    30  \
                 40         41         42         49         50         51   
              EP_mV      EP_mV      EP_mV      EP_mV      EP_mV      EP_mV   
Depth                                                                        
-2000.0   32.678833  32.524109  32.337494  33.458252  32.627869  32.371216   
-1500.0   32.727661  32.563477  32.366791  33.439941  32.686768  32.401733   
-1000.0   32.736511  32.604370  32.419281  33.425598  32.708740  32.422180   
-900.0          NaN        NaN        NaN        NaN        NaN        NaN   
-700.0          NaN        NaN        NaN        NaN        NaN        NaN   
...             ...        ...        ...        ...        ...        ...   
 29400.0        NaN        NaN        NaN        NaN        NaN        NaN   
 29500.0  32.887878  32.952881  32.241974  28.515625  31.864014  31.784058   
 29700.0        NaN        NaN        NaN        NaN        NaN        NaN   
 30000.0  32.856445  32.925415  32.111359  28.234253  31.782227  31.815491   
 NaN            NaN        NaN        NaN        NaN        NaN        NaN   

                                           
                 52         53         54  
              EP_mV      EP_mV      EP_mV  
Depth                                      
-2000.0   33.772583  33.246460  33.132019  
-1500.0   33.737183  33.324280  33.198547  
-1000.0   33.723450  33.408203  33.283386  
-900.0          NaN        NaN        NaN  
-700.0          NaN        NaN        NaN  
...             ...        ...        ...  
 29400.0        NaN        NaN        NaN  
 29500.0  33.638611  33.924866  33.811035  
 29700.0        NaN        NaN        NaN  
 30000.0  33.689880  3

In [63]:
# 1st datapoint --> the most positive one
ls_1st = list()
for c in df_ep.columns:
    ls_1st.append(df_ep[c].loc[df_ep[c].dropna().index[0]])

df = pd.DataFrame(ls_1st, columns=['1st datapoint'])

# --------------------------------------------------
# remove outlier --> z score
#q75,q25 = np.percentile(df, q=[75,25])
#intr_qr = q75-q25

#max = q75+(1.5*intr_qr)
#min = q25-(1.5*intr_qr)

#df[df < min] = np.nan
#df[df > max] = np.nan

df = df.dropna()

In [28]:
# polynomial fit of 2nd order: ax**2 + bx + c = y
paraFit = np.polyfit(df.index, df['1st datapoint'].to_numpy(), deg=2)
paraFit

array([ 1.84696570e-02, -5.24586863e-01,  3.62674129e+01])

In [52]:
# drift correction
df_ep_corr = df_ep.copy()

i0 = 0
for i, c in enumerate(df_ep_corr.columns):
    df_ep_corr[c[0]][c[1]].loc[:] = (df_ep[c[0]][c[1]] + (paraFit[0]*(i**2 - i0**2) + paraFit[1]*(i - i0))).values

In [62]:
# corrected EP profiles
df_ep_corr

35                                          26             \
                 13         14         15         16         17         18   
              EP_mV      EP_mV      EP_mV      EP_mV      EP_mV      EP_mV   
Depth                                                                        
-2000.0         NaN        NaN        NaN        NaN        NaN        NaN   
-1500.0         NaN        NaN        NaN        NaN        NaN        NaN   
-1000.0   39.589539  33.876512  33.514451  32.072538  31.984856  31.351533   
-900.0          NaN        NaN        NaN        NaN        NaN        NaN   
-700.0          NaN        NaN        NaN        NaN        NaN        NaN   
...             ...        ...        ...        ...        ...        ...   
 29400.0        NaN        NaN        NaN        NaN        NaN        NaN   
 29500.0  36.677856  35.086229  34.208726  32.555174  28.148186  48.254609   
 29700.0        NaN        NaN        NaN        NaN        NaN        NaN   
 30000.0  36.549988  35.122850  34.223374  32.559446  28.294976  50.974793   
 NaN            NaN        NaN        NaN        NaN        NaN        NaN   

                                      16             ...         15  \
                19         20         27         28  ...         39   
             EP_mV      EP_mV      EP_mV      EP_mV  ...      EP_mV   
Depth                                                ...              
-2000.0        NaN        NaN        NaN        NaN  ...  29.103256   
-1500.0        NaN        NaN        NaN        NaN  ...  29.111801   
-1000.0   30.92681  30.563441  31.257366  29.882671  ...  29.130112   
-900.0         NaN        NaN        NaN        NaN  ...        NaN   
-700.0         NaN        NaN        NaN  29.877483  ...        NaN   
...            ...        ...        ...        ...  ...        ...   
 29400.0       NaN        NaN        NaN        NaN  ...        NaN   
 29500.0  26.36550  29.833003  28.582653  27.215434  ...  29.480149   
 29700.0       NaN        NaN        NaN        NaN  ...        NaN   
 30000.0  25.89553  29.786464  28.286633  27.002116  ...  29.457260   
 NaN           NaN        NaN        NaN        NaN  ...        NaN   

                                                  23                    30  \
                 40         41         42         49         50         51   
              EP_mV      EP_mV      EP_mV      EP_mV      EP_mV      EP_mV   
Depth                                                                        
-2000.0   28.954670  28.810979  28.672336  29.878006  29.169474  29.071612   
-1500.0   29.003498  28.850346  28.701633  29.859696  29.228373  29.102129   
-1000.0   29.012348  28.891240  28.754123  29.845352  29.250346  29.122576   
-900.0          NaN        NaN        NaN        NaN        NaN        NaN   
-700.0          NaN        NaN        NaN        NaN        NaN        NaN   
...             ...        ...        ...        ...        ...        ...   
 29400.0        NaN        NaN        NaN        NaN        NaN        NaN   
 29500.0  29.163715  29.239751  28.576816  24.935379  28.405619  28.484453   
 29700.0        NaN        NaN        NaN        NaN        NaN        NaN   
 30000.0  29.132282  29.212285  28.446201  24.654007  28.323832  28.515887   
 NaN            NaN        NaN        NaN        NaN        NaN        NaN   

                                           
                 52         53         54  
              EP_mV      EP_mV      EP_mV  
Depth                                      
-2000.0   30.668709  30.375255  30.530422  
-1500.0   30.633308  30.453074  30.596950  
-1000.0   30.619575  30.536998  30.681789  
-900.0          NaN        NaN        NaN  
-700.0          NaN        NaN        NaN  
...             ...        ...        ...  
 29400.0        NaN        NaN        NaN  
 29500.0  30.534736  31.053660  31.209438  
 29700.0        NaN        NaN        NaN  
 30000.0  30.586006  31.068919  31.26

In [60]:
# showcast an example
fig, ax = plt.subplots()
ax.plot(df_ep[35][15].dropna()['EP_mV'].to_numpy(), df_ep[35][15].dropna().index, ls='-.')
ax.plot(df_ep_corr[35][15].dropna()['EP_mV'].to_numpy(), df_ep_corr[35][15].dropna().index)
ax.invert_yaxis(), plt.tight_layout(), sns.despine()

(None, None, None)